# Chapter 4: Full Experimental Pipeline for Approximate EASE

Этот notebook реализует полный воспроизводимый протокол экспериментальной главы 4 по приближённому обучению EASE.

## 0. Purpose and assumptions

В этой работе исследуется приближённое обучение EASE через решение систем $Gp_i=e_i$ без явного формирования плотных $G$, $G^{-1}$ и полного $W$.

- Подбор гиперпараметров выполняется только на validation.
- Используется последовательный операционный протокол: epsilon -> nystrom_rank -> block_size.
- После выбора $\epsilon^*$, $k^*$, $b^*$ выполняется одна итоговая оценка на test.
- Внутренние варианты EASE являются основным объектом сравнения.
- Внешние baseline-методы используются как контекстный ориентир качества.
- Notebook генерирует CSV, таблицы, графики, подписи, текстовые блоки и финальную проверку артефактов.

In [1]:
## 1. Environment and imports

import json
import math
import os
import platform
import sys
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy

try:
    import sklearn
    SKLEARN_VERSION = sklearn.__version__
except Exception:
    SKLEARN_VERSION = None

try:
    import implicit
    IMPLICIT_VERSION = implicit.__version__
except Exception:
    IMPLICIT_VERSION = None

ROOT = Path.cwd()
if not (ROOT / "experiments").exists() and (ROOT.parent / "experiments").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
EXPERIMENTS = ROOT / "experiments"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if str(EXPERIMENTS) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS))

from common import (
    dataset_stats_for_config,
    estimate_experiment_memory_gb,
    fit_and_evaluate,
    load_base_config,
    load_split,
    prepare_run_dir,
    save_yaml,
    single_threaded,
)
from iterative_ease.baselines import ALSImplicitRecommender, ItemKNNRecommender, PopularityRecommender
from iterative_ease.memory import current_memory_gb
from iterative_ease.metrics import evaluate_recommender_topk

from IPython.display import Markdown, display

plt.rcParams["axes.grid"] = True
plt.rcParams["figure.figsize"] = (7, 4)


def get_git_commit() -> str | None:
    import subprocess

    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return None


def get_ram_gb() -> float | None:
    try:
        if sys.platform == "darwin":
            import subprocess

            out = subprocess.check_output(["sysctl", "-n", "hw.memsize"], text=True).strip()
            return round(int(out) / (1024**3), 2)
    except Exception:
        pass
    return None


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def ensure_dirs(run_dir: Path) -> None:
    (run_dir / "tables").mkdir(parents=True, exist_ok=True)
    (run_dir / "figures").mkdir(parents=True, exist_ok=True)
    (run_dir / "text_blocks").mkdir(parents=True, exist_ok=True)


def save_df_csv_tex(df: pd.DataFrame, csv_path: Path, tex_path: Path, caption: str, label: str) -> None:
    df.to_csv(csv_path, index=False)
    write_text(tex_path, df.to_latex(index=False, escape=False, caption=caption, label=label))


def append_caption_snippet(path: Path, rel_fig_path: str, caption: str, label: str) -> None:
    snippet = (
        "\\begin{figure}[H]\n"
        "    \\centering\n"
        f"    \\includegraphics[width=0.85\\textwidth]{{{rel_fig_path}}}\n"
        f"    \\caption{{{caption}}}\n"
        f"    \\label{{{label}}}\n"
        "\\end{figure}\n\n"
    )
    with open(path, "a", encoding="utf-8") as f:
        f.write(snippet)


env_info = {
    "python_version": sys.version.split()[0],
    "numpy_version": np.__version__,
    "scipy_version": scipy.__version__,
    "pandas_version": pd.__version__,
    "sklearn_version": SKLEARN_VERSION,
    "implicit_version": IMPLICIT_VERSION,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "ram_gb": get_ram_gb(),
    "OMP_NUM_THREADS": os.environ.get("OMP_NUM_THREADS"),
    "OPENBLAS_NUM_THREADS": os.environ.get("OPENBLAS_NUM_THREADS"),
    "MKL_NUM_THREADS": os.environ.get("MKL_NUM_THREADS"),
    "VECLIB_MAXIMUM_THREADS": os.environ.get("VECLIB_MAXIMUM_THREADS"),
    "git_commit": get_git_commit(),
}

display(pd.DataFrame([env_info]))
print(json.dumps(env_info, ensure_ascii=False, indent=2))

,python_version,numpy_version,scipy_version,pandas_version,sklearn_version,implicit_version,platform,machine,processor,ram_gb,OMP_NUM_THREADS,OPENBLAS_NUM_THREADS,MKL_NUM_THREADS,VECLIB_MAXIMUM_THREADS,git_commit
0,3.12.5,1.26.4,1.11.4,2.1.4,1.3.2,None,macOS-26.3.1-arm64-arm-64bit,arm64,arm,16.0,None,None,None,None,8da71c64ede462053a8aad8fc06e8f9d0461d72b


{
  "python_version": "3.12.5",
  "numpy_version": "1.26.4",
  "scipy_version": "1.11.4",
  "pandas_version": "2.1.4",
  "sklearn_version": "1.3.2",
  "implicit_version": null,
  "platform": "macOS-26.3.1-arm64-arm-64bit",
  "machine": "arm64",
  "processor": "arm",
  "ram_gb": 16.0,
  "OMP_NUM_THREADS": null,
  "OPENBLAS_NUM_THREADS": null,
  "MKL_NUM_THREADS": null,
  "VECLIB_MAXIMUM_THREADS": null,
  "git_commit": "8da71c64ede462053a8aad8fc06e8f9d0461d72b"
}


## 2. Configuration

Загружается базовый YAML-конфиг и расширяется параметрами главы 4:
`seed`, `lambda`, `K`, `top_l`, `max_iter`, `q_min`, `tau_recall`, `tau_ndcg`, `memory_limit_gb`, `epsilon_grid`, `rank_grid`, `block_grid`.

## 3. Dataset loading and split

Строится разбиение train / validation / test. Для подбора гиперпараметров используется только validation.

## 4. Experiment 1 — Epsilon sweep on validation

**Цель.** Проверить влияние `epsilon` на время решения, число итераций, `q_epsilon`, Recall@K и NDCG@K.

**Split.** Validation only.

**Fixed parameters.** `block_size=4`, `nystrom_rank=32`.

**Varied parameter.** `epsilon in {1e-1, 1e-2, 1e-3, 1e-4}`.

**Selection rule.** Опорный запуск — минимальный epsilon из сетки; среди допустимых по quality/q_epsilon выбирается минимум `t_solve`.

## 5. Experiment 2 — Rank sweep on validation

**Цель.** Оценить влияние `nystrom_rank` на `t_build`, `t_solve`, `t_total`, итерации и качество.

**Split.** Validation only.

**Fixed parameters.** `epsilon=epsilon_star`, `block_size=4`.

**Varied parameter.** `nystrom_rank in {0, 32, 64, 128, 256}`.

**Selection rule.** Опорный запуск — максимальный успешный rank; среди допустимых выбирается минимум `t_total`.

## 6. Experiment 3 — Block size sweep on validation

**Цель.** Проверить влияние `block_size` на время, память, устойчивость и качество.

**Split.** Validation only.

**Fixed parameters.** `epsilon=epsilon_star`, `nystrom_rank=rank_star`.

**Varied parameter.** `block_size in {1, 4, 8, 16, 32}`.

**Selection rule.** Среди допустимых по quality/q_epsilon/memory/breakdown выбирается минимум `t_total`.

## 7. Selected configuration

Сохраняется `selected_config.yaml` с полями `selected_by: validation` и `selection_protocol: epsilon_then_rank_then_block_size`.

## 8. Experiment 4 — Final selected EASE on test

Выбранная на validation конфигурация оценивается один раз на test.

## 9. Experiment 5 — Internal EASE baselines on test

Сравниваются: `CG b=1`, `Block CG`, `Block PCG fixed`, `Block PCG selected`.

## 10. Experiment 6 — External baselines on test

Сравниваются: `Popularity`, `ItemKNN`, `ALS implicit` (с guard-логикой пропусков).

In [2]:
## 2-10. Methodologically corrected protocol (single notebook)

import threading
import psutil
from scipy import sparse

CONFIG_PATH = str(ROOT / "experiments/configs/default_m2.yaml")
base = load_base_config(CONFIG_PATH)
seed = int(base["data"].get("seed", 42))
np.random.seed(seed)

# strict requirement: no silent synthetic fallback
data_path = Path(base["data"]["path"])
if not data_path.is_absolute():
    data_path = ROOT / data_path
assert data_path.exists(), f"MovieLens data file not found: {data_path}"
base["data"]["path"] = str(data_path)

chapter_cfg = {
    "seed": seed,
    "lambda": float(base["model"]["reg"]),
    "K": int(base["model"]["k_recommend"]),
    "top_l": int(base["model"]["top_l"]) if base["model"].get("top_l") is not None else None,
    "max_iter": int(base["model"]["max_iter"]),
    "q_min": float(base.get("selection", {}).get("q_min", 0.95)),
    "tau_recall": float(base.get("selection", {}).get("tau_recall", 0.005)),
    "tau_ndcg": float(base.get("selection", {}).get("tau_ndcg", 0.005)),
    "memory_limit_gb": float(base.get("runtime", {}).get("max_memory_gb", 10.0)),
    "epsilon_grid": [1e-1, 1e-2, 1e-3, 1e-4],
    "rank_grid": [0, 16, 32, 64, 128, 256],
    "block_grid": [1, 2, 4, 8, 16, 32],
    "top_l_grid": [50, 100, 200, 500, 1000, None],
    "breakdown_threshold": 0,
    "n_runs": 1,
}

run_root = ROOT / "results" / "chapter4"
run_root.mkdir(parents=True, exist_ok=True)
with single_threaded():
    run_dir = prepare_run_dir(base, output_dir=str(run_root))
ensure_dirs(run_dir)

X_train, X_valid, X_test = load_split(base)
X_train_valid = (X_train + X_valid).tocsr()
X_total = (X_train + X_valid + X_test).tocsr()
M, N = X_total.shape


def rss_gb():
    return psutil.Process().memory_info().rss / (1024**3)


def run_with_peak(fn, *args, **kwargs):
    stop = threading.Event()
    peak = {"v": rss_gb()}

    def sampler():
        while not stop.is_set():
            v = rss_gb()
            if v > peak["v"]:
                peak["v"] = v
            stop.wait(0.02)

    t = threading.Thread(target=sampler, daemon=True)
    before = rss_gb()
    t.start()
    try:
        out = fn(*args, **kwargs)
    finally:
        stop.set()
        t.join(timeout=1)
    after = rss_gb()
    return out, before, after, peak["v"]


def evaluator_with_mask(W, X_mask, X_eval, K):
    t0 = time.perf_counter()
    from iterative_ease.metrics import evaluate_topk

    m = evaluate_topk(
        W=W,
        X_train=X_mask,
        X_test=X_eval,
        k=int(K),
        user_batch_size=int(base["runtime"]["user_batch_size"]),
        item_batch_size=base["runtime"].get("item_batch_size"),
    )
    return m, time.perf_counter() - t0


def fit_exact_ease(X_fit, lambda_, top_l=None):
    t0 = time.perf_counter()
    G = (X_fit.T @ X_fit).toarray().astype(np.float64)
    np.fill_diagonal(G, G.diagonal() + float(lambda_))
    n_items = G.shape[0]
    P = np.linalg.solve(G, np.eye(n_items, dtype=np.float64))
    diag = np.diag(P).copy()
    W = -(P / diag[None, :])
    np.fill_diagonal(W, 0.0)
    fit_t = time.perf_counter() - t0
    if top_l is None:
        return W, {"fit_time": fit_t, "diag_error": float(np.max(np.abs(np.diag(W))))}

    # truncate exact weights as separate approximation axis
    rows=[]; cols=[]; vals=[]
    for j in range(W.shape[1]):
        col = W[:, j]
        idx = np.argpartition(np.abs(col), -min(top_l, len(col)))[-min(top_l, len(col)):]
        nz = idx[col[idx] != 0.0]
        rows.extend(nz.tolist()); cols.extend([j]*len(nz)); vals.extend(col[nz].tolist())
    W_sp = sparse.csc_matrix((vals, (rows, cols)), shape=W.shape)
    return W_sp, {"fit_time": fit_t, "diag_error": 0.0}


def iterative_once(X_fit, X_mask_for_eval, X_eval, epsilon, rank, block_size, top_l):
    rec = {
        "epsilon": float(epsilon), "nystrom_rank": int(rank), "block_size": int(block_size), "top_l": -1 if top_l is None else int(top_l),
        "lambda": chapter_cfg["lambda"], "K": chapter_cfg["K"], "seed": chapter_cfg["seed"], "max_iter": chapter_cfg["max_iter"],
    }

    local = dict(base)
    local["model"] = dict(base["model"])
    local["model"]["top_l"] = top_l

    def _fit():
        return fit_and_evaluate(
            local, X_fit, X_eval,
            epsilon=float(epsilon),
            block_size=int(block_size),
            nystrom_rank=int(rank),
            nystrom_oversample=int(base["solver"].get("nystrom_oversample", 5)),
            show_progress=False,
        )

    t_all0 = time.perf_counter()
    (out, mem_before, mem_after, peak) = run_with_peak(_fit)
    W, row = out
    metrics, t_eval = evaluator_with_mask(W, X_mask_for_eval, X_eval, chapter_cfg["K"])
    t_all = time.perf_counter() - t_all0

    rec.update({
        "t_build_preconditioner": float(row.get("t_build", np.nan)),
        "t_solve": float(row.get("t_solve", np.nan)),
        "t_weight_build": float(row.get("t_total", np.nan)) - float(row.get("t_build", 0.0)) - float(row.get("t_solve", 0.0)),
        "t_recommend": float(t_eval),
        "t_eval": float(t_eval),
        "t_total": float(t_all),
        "q_epsilon": float(row.get("q_epsilon", np.nan)),
        "mean_residual": float(row.get("mean_residual", np.nan)),
        "max_residual": float(row.get("max_residual", np.nan)),
        "mean_iterations": float(row.get("mean_iterations", np.nan)),
        "median_iterations": float(row.get("median_iterations", np.nan)),
        "max_iterations": float(row.get("max_iterations", np.nan)),
        "p90_iterations": float(row.get("p90_iterations", np.nan)),
        "p95_iterations": float(row.get("p95_iterations", np.nan)),
        "total_breakdowns": int(row.get("total_breakdowns", 0)),
        "Recall@K": float(metrics["recall"]),
        "NDCG@K": float(metrics["ndcg"]),
        "memory_before_gb": float(mem_before),
        "memory_after_gb": float(mem_after),
        "peak_memory_gb": float(peak),
        "delta_memory_gb": float(mem_after - mem_before),
        "status": "ok",
        "warning": "",
    })
    return rec, W


def repeat_iterative(experiment_name, split, X_fit, X_mask_for_eval, X_eval, epsilon, rank, block_size, top_l, n_runs=1):
    rows=[]; lastW=None
    for run in range(n_runs):
        r, w = iterative_once(X_fit, X_mask_for_eval, X_eval, epsilon, rank, block_size, top_l)
        r["experiment_name"]=experiment_name
        r["split"]=split
        r["run_id"]=run
        rows.append(r); lastW=w
    df = pd.DataFrame(rows)
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != "run_id"]
    out = {"experiment_name": experiment_name, "split": split}
    for c in num_cols:
        s = df[c]
        out[f"{c}_mean"] = float(s.mean())
        out[f"{c}_median"] = float(s.median())
        out[f"{c}_std"] = float(s.std(ddof=1)) if len(s) > 1 else 0.0
        out[f"{c}_min"] = float(s.min())
        out[f"{c}_max"] = float(s.max())
    # keep representative params
    out.update({"epsilon":epsilon,"nystrom_rank":rank,"block_size":block_size,"top_l":top_l,"n_runs":n_runs})
    out["status"]="ok"
    out["warning"]="Timing is single-run and approximate." if n_runs==1 else ""
    return pd.DataFrame(rows), out, lastW


# Reference models
exact_full_W, exact_full_meta = fit_exact_ease(X_train, chapter_cfg["lambda"], top_l=None)
exact_trunc_W, exact_trunc_meta = fit_exact_ease(X_train, chapter_cfg["lambda"], top_l=chapter_cfg["top_l"])
exact_valid, t_eval_exact_valid = evaluator_with_mask(exact_full_W, X_train, X_valid, chapter_cfg["K"])
exact_test, t_eval_exact_test = evaluator_with_mask(exact_full_W, X_train, X_test, chapter_cfg["K"])

reference_quality = pd.DataFrame([
    {"method":"Exact dense EASE full W", "split":"validation", "Recall@20":exact_valid["recall"], "NDCG@20":exact_valid["ndcg"], "fit_time":exact_full_meta["fit_time"], "eval_time":t_eval_exact_valid},
    {"method":"Exact dense EASE full W", "split":"test", "Recall@20":exact_test["recall"], "NDCG@20":exact_test["ndcg"], "fit_time":exact_full_meta["fit_time"], "eval_time":t_eval_exact_test},
])

# epsilon sweep (validation)
eps_rows=[]; eps_raw=[]
for eps in chapter_cfg["epsilon_grid"]:
    raw, summary, W = repeat_iterative("epsilon_sweep", "validation", X_train, X_train, X_valid, eps, 32, 4, chapter_cfg["top_l"], chapter_cfg["n_runs"])
    summary["abs_recall_drop_vs_exact"] = exact_valid["recall"] - summary.get("Recall@K_mean", np.nan)
    summary["rel_recall_drop_vs_exact"] = summary["abs_recall_drop_vs_exact"] / max(exact_valid["recall"], 1e-12)
    summary["abs_ndcg_drop_vs_exact"] = exact_valid["ndcg"] - summary.get("NDCG@K_mean", np.nan)
    summary["rel_ndcg_drop_vs_exact"] = summary["abs_ndcg_drop_vs_exact"] / max(exact_valid["ndcg"], 1e-12)
    eps_rows.append(summary); eps_raw.append(raw)
epsilon_table = pd.DataFrame(eps_rows).sort_values("epsilon")
epsilon_raw = pd.concat(eps_raw, ignore_index=True)

# select epsilon: fastest acceptable
ref_eps_row = epsilon_table.sort_values("epsilon").iloc[0]
allowed = epsilon_table[(epsilon_table["Recall@K_mean"] >= ref_eps_row["Recall@K_mean"] - chapter_cfg["tau_recall"]) &
                        (epsilon_table["NDCG@K_mean"] >= ref_eps_row["NDCG@K_mean"] - chapter_cfg["tau_ndcg"]) &
                        (epsilon_table["q_epsilon_mean"] >= chapter_cfg["q_min"]) ]
if allowed.empty:
    eps_star = float(ref_eps_row["epsilon"])
else:
    eps_star = float(allowed.sort_values("t_solve_mean").iloc[0]["epsilon"])

# rank sweep
rank_rows=[]
for rk in chapter_cfg["rank_grid"]:
    _, summary, _ = repeat_iterative("rank_sweep", "validation", X_train, X_train, X_valid, eps_star, rk, 4, chapter_cfg["top_l"], chapter_cfg["n_runs"])
    rank_rows.append(summary)
rank_table = pd.DataFrame(rank_rows).sort_values("nystrom_rank")
rank_ref = rank_table.sort_values("nystrom_rank").iloc[-1]
rank_allowed = rank_table[(rank_table["Recall@K_mean"] >= rank_ref["Recall@K_mean"] - chapter_cfg["tau_recall"]) &
                          (rank_table["NDCG@K_mean"] >= rank_ref["NDCG@K_mean"] - chapter_cfg["tau_ndcg"]) &
                          (rank_table["q_epsilon_mean"] >= chapter_cfg["q_min"]) ]
rank_star = int((rank_allowed if not rank_allowed.empty else rank_table).sort_values("t_total_mean").iloc[0]["nystrom_rank"])

# block sweep
block_rows=[]
for b in chapter_cfg["block_grid"]:
    _, summary, _ = repeat_iterative("block_sweep", "validation", X_train, X_train, X_valid, eps_star, rank_star, b, chapter_cfg["top_l"], chapter_cfg["n_runs"])
    block_rows.append(summary)
block_table = pd.DataFrame(block_rows).sort_values("block_size")
block_ref = block_table.sort_values("block_size").iloc[-1]
block_allowed = block_table[(block_table["Recall@K_mean"] >= block_ref["Recall@K_mean"] - chapter_cfg["tau_recall"]) &
                            (block_table["NDCG@K_mean"] >= block_ref["NDCG@K_mean"] - chapter_cfg["tau_ndcg"]) &
                            (block_table["q_epsilon_mean"] >= chapter_cfg["q_min"]) &
                            (block_table["total_breakdowns_mean"] <= chapter_cfg["breakdown_threshold"]) ]
block_star = int((block_allowed if not block_allowed.empty else block_table).sort_values("t_total_mean").iloc[0]["block_size"])

# top_l sweep
topl_rows=[]
for top_l in chapter_cfg["top_l_grid"]:
    _, summary, _ = repeat_iterative("topl_sweep", "validation", X_train, X_train, X_valid, eps_star, rank_star, block_star, top_l, chapter_cfg["n_runs"])
    topl_rows.append(summary)
topl_table = pd.DataFrame(topl_rows)

# strict iterative vs selected
_, strict_summary, strict_W = repeat_iterative("iterative_strict", "validation", X_train, X_train, X_valid, 1e-4, 0, 1, None, chapter_cfg["n_runs"])
_, selected_val_summary, selected_val_W = repeat_iterative("iterative_selected", "validation", X_train, X_train, X_valid, eps_star, rank_star, block_star, chapter_cfg["top_l"], chapter_cfg["n_runs"])

# final test protocol A/B
_, finalA_summary, finalA_W = repeat_iterative("final_test_train_only", "test", X_train, X_train, X_test, eps_star, rank_star, block_star, chapter_cfg["top_l"], chapter_cfg["n_runs"])
_, finalB_summary, finalB_W = repeat_iterative("final_test_refit_train_valid", "test", X_train_valid, X_train_valid, X_test, eps_star, rank_star, block_star, chapter_cfg["top_l"], chapter_cfg["n_runs"])

# W / topK stability wrt exact
def overlap_at_k(Wa, Wb, X_mask, k=20, users=500):
    from iterative_ease.metrics import _topk_full
    u=min(users, X_mask.shape[0])
    A = _topk_full(Wa, X_mask[:u], k)
    B = _topk_full(Wb, X_mask[:u], k)
    inter=[len(set(a).intersection(set(b)))/k for a,b in zip(A,B)]
    changed=[1.0 if set(a)!=set(b) else 0.0 for a,b in zip(A,B)]
    return float(np.mean(inter)), float(np.mean(changed))

W_compare=[]
if isinstance(exact_full_W, np.ndarray) and hasattr(finalA_W, 'toarray'):
    Wapp=finalA_W.toarray()
    denom=np.linalg.norm(exact_full_W) + 1e-12
    diff= Wapp - exact_full_W
    fro=float(np.linalg.norm(diff)/denom)
    max_abs=float(np.max(np.abs(diff)))
    mean_abs=float(np.mean(np.abs(diff)))
    diag_err=float(np.max(np.abs(np.diag(Wapp))))
else:
    fro=max_abs=mean_abs=np.nan
    diag_err = float(np.max(np.abs(finalA_W.diagonal()))) if sparse.issparse(finalA_W) else np.nan

ovlp, changed = overlap_at_k(exact_full_W if isinstance(exact_full_W,np.ndarray) else exact_full_W.toarray(), finalA_W if isinstance(finalA_W,np.ndarray) else finalA_W.toarray(), X_train, k=chapter_cfg["K"], users=500)
W_compare.append({"model":"Iterative selected vs Exact", "relative_fro_error":fro, "max_abs_weight_error":max_abs, "mean_abs_weight_error":mean_abs, "diag_error":diag_err, "topK_overlap_with_exact":ovlp, "percentage_users_with_changed_topK":changed})
W_compare_df=pd.DataFrame(W_compare)

# sanity checks
assert np.max(np.abs(np.diag(exact_full_W))) < 1e-8, "diag(W_exact) not zero"
assert finalA_summary["q_epsilon_mean"] <= 1.0 + 1e-9
assert np.isfinite(epsilon_table["Recall@K_mean"]).all()

# potential non-informative iteration stats warning
iter_cols = epsilon_table[["mean_iterations_mean", "median_iterations_mean", "max_iterations_mean"]]
if ((iter_cols["median_iterations_mean"] == iter_cols["mean_iterations_mean"]) & (iter_cols["max_iterations_mean"] == iter_cols["mean_iterations_mean"])).all():
    print("WARNING: iteration statistics may be non-informative (median=max=mean across all epsilon rows)")

# tables and outputs
reference_quality.to_csv(run_dir / "tables/reference_quality_table.csv", index=False)
epsilon_table.to_csv(run_dir / "tables/epsilon_sweep_table.csv", index=False)
rank_table.to_csv(run_dir / "tables/rank_sweep_table.csv", index=False)
block_table.to_csv(run_dir / "tables/block_sweep_table.csv", index=False)
topl_table.to_csv(run_dir / "tables/topl_sweep_table.csv", index=False)
W_compare_df.to_csv(run_dir / "tables/w_stability_table.csv", index=False)

final_protocol = pd.DataFrame([
    {"protocol":"A_train_only", **finalA_summary},
    {"protocol":"B_refit_train_plus_valid", **finalB_summary},
])
final_protocol.to_csv(run_dir / "tables/final_test_protocol_table.csv", index=False)

# core csv outputs expected by earlier cells
epsilon_table.to_csv(run_dir / "epsilon_sweep.csv", index=False)
rank_table.to_csv(run_dir / "rank_sweep.csv", index=False)
block_table.to_csv(run_dir / "block_sweep.csv", index=False)
final_protocol.to_csv(run_dir / "final_selected_test.csv", index=False)

selected_cfg = {
    "epsilon": eps_star, "nystrom_rank": rank_star, "block_size": block_star,
    "lambda": chapter_cfg["lambda"], "top_l": chapter_cfg["top_l"], "K": chapter_cfg["K"],
    "seed": chapter_cfg["seed"], "max_iter": chapter_cfg["max_iter"],
    "q_min": chapter_cfg["q_min"], "tau_recall": chapter_cfg["tau_recall"], "tau_ndcg": chapter_cfg["tau_ndcg"],
    "selected_by": "validation", "selection_protocol": "epsilon_then_rank_then_block_size",
}
save_yaml(run_dir / "selected_config.yaml", selected_cfg)

# compact in-notebook displays
display(Markdown("## Corrected Core Tables"))
display(reference_quality)
display(epsilon_table[["epsilon","Recall@K_mean","NDCG@K_mean","t_solve_mean","q_epsilon_mean","abs_recall_drop_vs_exact","abs_ndcg_drop_vs_exact"]])
display(rank_table[["nystrom_rank","Recall@K_mean","NDCG@K_mean","t_build_preconditioner_mean","t_solve_mean","t_total_mean"]])
display(block_table[["block_size","Recall@K_mean","NDCG@K_mean","t_total_mean","peak_memory_gb_mean","q_epsilon_mean"]])
display(topl_table[["top_l","Recall@K_mean","NDCG@K_mean","t_total_mean"]])
display(final_protocol[["protocol","Recall@K_mean","NDCG@K_mean","t_total_mean"]])
display(W_compare_df)

summary_text = f"""
Selected configuration is not the highest-quality EASE approximation. It is the fastest configuration satisfying predefined validation tolerances tau_recall and tau_ndcg relative to the reference/strict run.

Observed quality drop is expected and is caused by the allowed residual tolerance, possible weight truncation top_l, and ranking instability near the Top-K boundary.

Block size and preconditioner rank should be interpreted primarily as computational parameters. They should not change the target EASE model; quality differences at fixed epsilon indicate residual numerical error or implementation-level approximation.

Selected params: epsilon={eps_star}, rank={rank_star}, block_size={block_star}.
Results directory: {run_dir}
"""
write_text(run_dir / "text_blocks/09_chapter4_conclusion.md", summary_text)
display(Markdown("## Honest Summary\n\n" + summary_text))

print("Run directory:", run_dir)


MemoryError: Refusing to build dense W for n_items > 3000. Set top_l to a positive integer.

In [ ]:
## 11. Text summaries and run manifest (updated)

from datetime import datetime

text_paths = sorted((run_dir / "text_blocks").glob("*.md"))
summary = [
    "# Chapter 4 Summary",
    "",
    "Protocol: epsilon -> rank -> block_size (validation only), then final test protocol A/B.",
    f"Selected epsilon={eps_star}, rank={rank_star}, block_size={block_star}",
    "",
    "## Generated tables",
]
for p in sorted((run_dir / "tables").glob("*")):
    summary.append(f"- {p.relative_to(run_dir)}")
summary.append("")
summary.append("## Generated figures")
for p in sorted((run_dir / "figures").glob("*")):
    summary.append(f"- {p.relative_to(run_dir)}")

write_text(run_dir / "chapter4_summary.md", "\n".join(summary) + "\n")

manifest = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "config_path": CONFIG_PATH,
    "dataset_path": str(data_path),
    "selected_config": selected_cfg,
    "run_dir": str(run_dir),
}
write_text(run_dir / "run_manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2))

# duplicate captions file for compatibility
if (run_dir / "chapter4_figures.tex").exists():
    write_text(run_dir / "captions.tex", (run_dir / "chapter4_figures.tex").read_text(encoding="utf-8"))
else:
    write_text(run_dir / "captions.tex", "")

print("summary + manifest saved")


In [ ]:
## 13. Inline results view (updated)

from IPython.display import Image, Markdown, display
print("Results directory:", run_dir)

for t in [
    "tables/reference_quality_table.csv",
    "tables/epsilon_sweep_table.csv",
    "tables/rank_sweep_table.csv",
    "tables/block_sweep_table.csv",
    "tables/topl_sweep_table.csv",
    "tables/final_test_protocol_table.csv",
    "tables/w_stability_table.csv",
]:
    p = run_dir / t
    if p.exists():
        display(Markdown(f"### {t}"))
        display(pd.read_csv(p))

for fig_name in sorted([p.name for p in (run_dir / "figures").glob("*.png")]):
    display(Markdown(f"### figures/{fig_name}"))
    display(Image(filename=str(run_dir / "figures" / fig_name)))

if (run_dir / "chapter4_summary.md").exists():
    display(Markdown("## chapter4_summary.md"))
    display(Markdown((run_dir / "chapter4_summary.md").read_text(encoding="utf-8")))


In [ ]:
## 12. Final checklist (updated)

must_exist = [
    "epsilon_sweep.csv",
    "rank_sweep.csv",
    "block_sweep.csv",
    "selected_config.yaml",
    "final_selected_test.csv",
    "run_manifest.json",
    "chapter4_summary.md",
    "tables/reference_quality_table.csv",
    "tables/epsilon_sweep_table.csv",
    "tables/rank_sweep_table.csv",
    "tables/block_sweep_table.csv",
    "tables/topl_sweep_table.csv",
    "tables/final_test_protocol_table.csv",
]
for rel in must_exist:
    assert (run_dir / rel).exists(), f"Missing required artifact: {rel}"

eps = pd.read_csv(run_dir / "tables/epsilon_sweep_table.csv")
rnk = pd.read_csv(run_dir / "tables/rank_sweep_table.csv")
blk = pd.read_csv(run_dir / "tables/block_sweep_table.csv")
assert (eps["split"] == "validation").all()
assert (rnk["split"] == "validation").all()
assert (blk["split"] == "validation").all()

finalp = pd.read_csv(run_dir / "tables/final_test_protocol_table.csv")
assert set(finalp["split"].unique()) == {"test"}
assert {"A_train_only","B_refit_train_plus_valid"}.issubset(set(finalp["protocol"]))

sel = selected_cfg
assert sel["selected_by"] == "validation"
assert sel["selection_protocol"] == "epsilon_then_rank_then_block_size"

print("All hard checks passed.")
print("run_dir:", run_dir)
